# Figure 11 -- variability and forecast departure on the SWOT ground track

Every pixel is a fixed point on a repeating ground track, so these are statistics
of one piece of ocean over the year rather than a mixture of places. The four
panels are the observed variability, the variability of the forecast departure,
their ratio, and the two variabilities against each other.

**What this reproduces.** All four panels. Every stored field is a standard
deviation over the repeats rather than a swath observation, so none of them had to
be withheld; the SWOT product itself is cited in the paper.

Shelf pixels are retained. The mean dynamic topography is least reliable there and
the ratio approaches or exceeds one along the shelves and in the Florida Straits;
the quoted statistics are medians and are insensitive to those values.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
import matplotlib.colors as mcolors

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    PROJ, HAVE_CARTOPY = ccrs.PlateCarree(), True
except ImportError:
    PROJ, HAVE_CARTOPY = None, False
    print("cartopy is not installed -- drawing the maps without coastlines")

# Cartopy fetches its coastline data from the network the first time it is used.
# Try once here, so a machine that is offline or has no cached copy degrades to a
# map without coastlines instead of failing in the middle of a figure.
HAVE_COAST = False
if HAVE_CARTOPY:
    try:
        next(iter(cfeature.COASTLINE.geometries()))
        HAVE_COAST = True
    except Exception as exc:
        print(f"cartopy cannot reach its coastline data ({type(exc).__name__}) -- "
              f"drawing the maps without coastlines")

TKW = dict(transform=PROJ) if HAVE_CARTOPY else {}


def map_features(ax, labels=True, land="lightgray", lw=0.4, zland=3, zcoast=4):
    """Coastlines and gridlines where available; plain axes otherwise."""
    if HAVE_COAST:
        ax.add_feature(cfeature.LAND, facecolor=land, zorder=zland)
        ax.add_feature(cfeature.COASTLINE, linewidth=lw, zorder=zcoast)
    if HAVE_CARTOPY:
        gl = ax.gridlines(draw_labels=labels, linestyle="--", lw=0.2)
        gl.top_labels = gl.right_labels = False
        return gl
    ax.tick_params(labelsize=7)
    return None
SIG_MAX_SWOT, SIG_MAX_DIFF, RATIO_MAX = 0.25, 0.12, 2.0
NBINS, MAX_POINTS = 150, 400_000

for path in need("Figure11_swot_repeat_statistics_*.nc"):
    with xr.open_dataset(path) as ds:
        lon = ds.longitude.values
        lon = np.where(lon > 180, lon - 360.0, lon)
        lat = ds.latitude.values
        sd  = ds.sigma_departure.values.astype("f8")
        sf  = ds.sigma_forecast.values.astype("f8")
        has_swot = "sigma_swot" in ds
        sw = ds.sigma_swot.values.astype("f8") if has_swot else None
        if not has_swot:
            print("sigma(SWOT) is not in this file: only the departure map "
                  "can be drawn")

        maps = []
        if has_swot:
            maps.append((r"$\sigma$(SWOT) (m)", sw, "viridis", 0, SIG_MAX_SWOT))
        maps.append((r"$\sigma$(forecast $-$ SWOT) (m)", sd, "magma", 0, SIG_MAX_DIFF))
        if has_swot:
            with np.errstate(invalid="ignore", divide="ignore"):
                maps.append(("ratio", sd / sw, "RdBu_r", None, None))

        fig = plt.figure(figsize=(5.0 * len(maps), 8.4), dpi=130)
        gs = fig.add_gridspec(2, 2 * len(maps), height_ratios=[1.0, 1.25],
                              hspace=0.25, wspace=0.55)
        for k, (ttl, c, cm, vmin, vmax) in enumerate(maps):
            ax = fig.add_subplot(gs[0, 2 * k:2 * k + 2],
                                 projection=PROJ) if HAVE_CARTOPY else \
                 fig.add_subplot(gs[0, 2 * k:2 * k + 2])
            norm = (mcolors.TwoSlopeNorm(vmin=0.0, vcenter=1.0, vmax=RATIO_MAX)
                    if vmin is None else None)
            s = ax.scatter(lon, lat, c=c, cmap=cm, s=0.35, norm=norm,
                           vmin=vmin, vmax=vmax, rasterized=True, zorder=2, **TKW)
            gl = map_features(ax, land="0.85", lw=0.5, zland=6, zcoast=7)
            if gl is not None:
                gl.left_labels = (k == 0)
                gl.xlabel_style = gl.ylabel_style = {"size": 8}
            cb = fig.colorbar(s, ax=ax, orientation="horizontal", pad=0.07,
                              fraction=0.045, extend="max")
            cb.set_label(ttl, fontsize=9)
            cb.ax.tick_params(labelsize=8)
            ax.set_title(ttl, fontsize=10, fontweight="bold")
            ax.text(0.0, 1.04, f"({chr(97 + k)})", transform=ax.transAxes,
                    fontsize=10, fontweight="bold", va="bottom")

        if has_swot:
            ax = fig.add_subplot(gs[1, len(maps) - 1:len(maps) + 1])
            ok = np.isfinite(sf) & np.isfinite(sw)
            x, y = sf[ok], sw[ok]
            if x.size > MAX_POINTS:
                sel = np.random.default_rng(0).choice(x.size, MAX_POINTS,
                                                      replace=False)
                xp, yp = x[sel], y[sel]
            else:
                xp, yp = x, y
            hi = float(np.percentile(np.r_[xp, yp], 99.5))
            cmin = 1.0 / (xp.size * (hi / NBINS) ** 2)
            h = ax.hist2d(xp, yp, bins=NBINS, range=[[0, hi], [0, hi]],
                          cmap="jet", density=True, cmin=cmin, rasterized=True)
            ax.plot([0, hi], [0, hi], "k--", lw=2.0, zorder=3)
            slope, icept = np.polyfit(x, y, 1)
            ax.plot([0, hi], [icept, slope * hi + icept], color="red", lw=1.4,
                    zorder=4)
            ax.set_xlim(0, hi); ax.set_ylim(0, hi); ax.set_aspect("equal")
            ax.set_xlabel(r"$\sigma$(forecast SSH) (m)", fontsize=10)
            ax.set_ylabel(r"$\sigma$(SWOT SSH) (m)", fontsize=10)
            ax.tick_params(labelsize=8)
            ax.text(0.03, 0.97,
                    f"Correlation: {np.corrcoef(x, y)[0, 1]:.2f}\n \n"
                    f"median ratio: {np.median(x / y):.2f}",
                    transform=ax.transAxes, va="top", ha="left", fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.35", facecolor="white",
                              alpha=0.85, edgecolor="0.6", linewidth=0.5))
            cb = fig.colorbar(h[3], ax=ax, orientation="vertical", pad=0.02,
                              fraction=0.045, extend="max")
            cb.set_label("Density (m$^{-2}$)", fontsize=9)
            cb.ax.tick_params(labelsize=8)
            ax.text(0.0, 1.03, f"({chr(97 + len(maps))})", transform=ax.transAxes,
                    fontsize=10, fontweight="bold", va="bottom")

        tag = path.stem.replace("Figure11_swot_repeat_statistics_", "")
        out = FIGDIR / f"Figure11_{tag}.png"
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("wrote", out)

        if has_swot:
            okr = np.isfinite(sd) & np.isfinite(sw)
            rt = sd[okr] / sw[okr]
            print(f"  sigma(SWOT)           median {np.median(sw[okr]):.3f} m")
            print(f"  sigma(forecast-SWOT)  median {np.median(sd[okr]):.3f} m")
            print(f"  ratio                 median {np.median(rt):.2f}")
            print(f"  pixels with ratio > 1: {100 * np.mean(rt > 1):.1f}%")
        print(f"  anomaly correlation {ds.attrs['anomaly_correlation']}, "
              f"amplitude ratio {ds.attrs['anomaly_amplitude_ratio']}")
